# Few shot NABirds classification

This notebook's objective is to build a few shot classifier model to achieve the highest accuracy possible on the NABirds dataset. Iterative pseudo-labeling is then used on the rest of the training data to attempt to increase the model's performance.

Accuracy is the chosen metric here, as the dataset is relatively balanced. As some imbalance is present, metrics such as precision, recall, and F1 score are tracked. As no separate test dataset is present, the validation set is used as hold-out test data, and only used at the very end to test the model's final performance. The goal accuracy for this problem is 70%.

Notebook structure:
- Setup and configuration
- Loading and splitting training data
- Multi-backbone feature extractor
- Initial few-shot baseline evaluation
- Iterative pseudo-labeling
- Prototypical network
- Model evaluation
- Conclusion

# Setup and configuration:

A GPU from Google Colab is used for this project, so relevant packages are installed in the environment, and relevant files (fewshot.py, label cleaning files) are linked from github:

In [ ]:
import sys
import urllib.request
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Torrtheb/data-science-portfolio.git"
BRANCH = "bird-classifier-clean"
RAW_BASE_URL = f"https://raw.githubusercontent.com/Torrtheb/data-science-portfolio/{BRANCH}/bird_classifier"

CLEANED_INDEX_FILES = [
    "label_index_train_clean.npz",
    "label_index_val_clean.npz",
    "clean_indices.npz",
]

PYTHON_MODULES = ["eda", "fewshot", "model_tune", "utilities_fewshot"]

if IN_COLAB:
    %pip install -q imagehash "deeplake<4" umap-learn albumentations \
    opencv-python-headless scikit-image lime scikit-learn seaborn \
    jupyter-black optuna

    REPO_PATH = "/content/repo"
    PROJECT_PATH = f"{REPO_PATH}/bird_classifier"
    MODULE_PATH = f"{PROJECT_PATH}/python_files"

    !rm -rf {REPO_PATH}
    !git clone --branch {BRANCH} --single-branch --quiet {REPO_URL} {REPO_PATH}

    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)

    for mod in PYTHON_MODULES:
        if mod in sys.modules:
            del sys.modules[mod]

    print("\n Downloading cleaned index files from GitHub...")
    for filename in CLEANED_INDEX_FILES:
        dest_path = Path(PROJECT_PATH) / filename
        url = f"{RAW_BASE_URL}/{filename}"
        try:
            urllib.request.urlretrieve(url, str(dest_path))
            print(f"   {filename}")
        except Exception as e:
            print(f"   {filename} not available (run eda.ipynb locally first): {e}")

    print("\n Colab setup complete")
else:
    MODULE_PATH = str(Path("./python_files").resolve())
    if MODULE_PATH not in sys.path:
        sys.path.insert(0, MODULE_PATH)
    print("Running locally")

Importing the rest of the libraries used in this notebook:

In [ ]:
import random
import warnings
from typing import Dict, List, Tuple, Optional, Any
from pathlib import Path
from collections import defaultdict
import jupyter_black
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torchvision import transforms as T
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import confusion_matrix
import seaborn as sns
from lime import lime_image
from skimage.segmentation import mark_boundaries
import deeplake
import cv2
from collections import Counter
import json
from datetime import datetime


from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    f1_score,
)

jupyter_black.load()

from fewshot import (
    get_device,
    build_label_index,
    load_label_index,
    save_label_index,
    flatten_indices,
    create_fewshot_split,
    MultiBackboneFeatureExtractor,
    visualize_preprocessing_modes,
    get_labels_for_indices,
    evaluate_backbone_fewshot,
    FewShotExperiment,
    spot_check_pseudo_labels,
)

from model_tune import BirdDataset
from utilities_fewshot import run_learned_projection_experiment

Initializing device, enabling CUDA optimizations, inititalizing batch size, reproducibility, data, caching, and experiment settings:

In [ ]:
DEVICE = get_device()
%matplotlib inline
%config InlineBackend.figure_format = 'png'
if torch.cuda.is_available():
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.1f} GB VRAM)")
    torch.backends.cudnn.benchmark = True
    if hasattr(torch.backends.cuda.matmul, "fp32_precision"):
        torch.backends.cuda.matmul.fp32_precision = "tf32"
    else:
        torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch.backends.cudnn, "conv") and hasattr(
        torch.backends.cudnn.conv, "fp32_precision"
    ):
        torch.backends.cudnn.conv.fp32_precision = "tf32"
    else:
        torch.backends.cudnn.allow_tf32 = True
    print("CUDA optimizations enabled (cuDNN benchmark, TF32)")

    if gpu_mem_gb >= 15:
        BATCH_SIZE = 128
    elif gpu_mem_gb >= 8:
        BATCH_SIZE = 64
    else:
        BATCH_SIZE = 32
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    BATCH_SIZE = 32
else:
    BATCH_SIZE = 32

print(f"Using device: {DEVICE} (batch size {BATCH_SIZE})")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


DATA_ROOT = Path("/content/data") if IN_COLAB else Path("data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

EMBEDDING_CACHE_DIR = DATA_ROOT / "embedding_cache"
EMBEDDING_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_CACHE_DIR = DATA_ROOT / "split_cache"
SPLIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR = DATA_ROOT / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

FEWSHOT_PER_CLASS = 5
CONFIDENCE_THRESHOLD = 0.9
USE_FP16_EMBEDDINGS = True
SKIP_BACKBONE_COMPARISON = False

print(f"\nData directory: {DATA_ROOT}")

# Loading and splitting training data:

The training data is now loaded from DeepLake, and duplicate images are removed as in eda.ipynb:

In [ ]:
ds_train = deeplake.load("hub://activeloop/nabirds-dataset-train", read_only=True)
print(f"Train samples (raw): {len(ds_train)}")

CLEAN_INDICES_PATHS = [
    Path(MODULE_PATH) / "clean_indices.npz",
    Path("clean_indices.npz"),
    DATA_ROOT / "clean_indices.npz",
]
_clean_npz = None
for p in CLEAN_INDICES_PATHS:
    if p.exists():
        _clean_npz = np.load(p)
        print(f"Loaded cleaned index file: {p}")
        break

CLEAN_TRAIN_INDICES = None
if _clean_npz is not None and "train_indices" in _clean_npz.files:
    CLEAN_TRAIN_INDICES = _clean_npz["train_indices"].astype(np.int64)
    print(
        f"Using cleaned train indices (duplicates dropped): {len(CLEAN_TRAIN_INDICES)} kept"
    )


LABEL_INDEX_CANDIDATES = [
    Path(MODULE_PATH) / "label_index_train_clean.npz",
    Path("label_index_train_clean.npz"),
    DATA_ROOT / "label_index_train_clean.npz",
    Path(MODULE_PATH) / "label_index_train.npz",
    Path("label_index_train.npz"),
    DATA_ROOT / "label_index_train.npz",
]
LABEL_INDEX_PATH = next((p for p in LABEL_INDEX_CANDIDATES if p.exists()), None)

if LABEL_INDEX_PATH is not None:
    label_index = load_label_index(LABEL_INDEX_PATH)
    print(f"Loaded label index from {LABEL_INDEX_PATH}")
else:
    label_index = build_label_index(ds_train)
    if CLEAN_TRAIN_INDICES is not None:
        allowed = set(map(int, CLEAN_TRAIN_INDICES))
        label_index = {
            int(class_id): np.array(
                [int(i) for i in indices if int(i) in allowed], dtype=np.int64
            )
            for class_id, indices in label_index.items()
        }

    out_path = DATA_ROOT / "label_index_train_clean.npz"
    save_label_index(label_index, out_path)
    print(f"Built and saved label index to {out_path}")

class_ids = np.array(sorted(label_index.keys()))
NUM_CLASSES = len(class_ids)
print(f"Number of classes: {NUM_CLASSES}")

The same number of classes, and training samples as in the eda notebook are seen. The validation data is used as test data (no separate test dataset is present for this dataset), and will not be loaded until model evaluation. Therefore, the training data is split into separate training (70%), validation (15%), and test (15%) data. Then, the training data is split into a support set (5 images maximum for all classes, as some classes have less than 5 images), and an 'unlabeled' pool with the rest of the training images.

In [ ]:
SPLIT_CACHE_PATH = (
    SPLIT_CACHE_DIR / f"fewshot_split_n{FEWSHOT_PER_CLASS}_seed{SEED}.npz"
)

if SPLIT_CACHE_PATH.exists():
    print(f"Loading cached splits from {SPLIT_CACHE_PATH}")
    with np.load(SPLIT_CACHE_PATH, allow_pickle=True) as data:
        support_indices = {int(k): v for k, v in data["support_indices"].item().items()}
        pool_indices = {int(k): v for k, v in data["pool_indices"].item().items()}
        val_indices_split = {int(k): v for k, v in data["val_indices"].item().items()}
        test_indices_split = {int(k): v for k, v in data["test_indices"].item().items()}
else:
    print("Creating new train/val/test splits...")
    support_indices, pool_indices, val_indices_split, test_indices_split = (
        create_fewshot_split(
            label_index,
            n_support=FEWSHOT_PER_CLASS,
            val_fraction=0.15,
            test_fraction=0.15,
            seed=SEED,
        )
    )
    np.savez(
        SPLIT_CACHE_PATH,
        support_indices=support_indices,
        pool_indices=pool_indices,
        val_indices=val_indices_split,
        test_indices=test_indices_split,
    )
    print(f"Saved splits to {SPLIT_CACHE_PATH}")

val_flat = flatten_indices(val_indices_split)
test_flat = flatten_indices(test_indices_split)

n_support_total = sum(len(v) for v in support_indices.values())
n_pool_total = sum(len(v) for v in pool_indices.values())
n_val_total = len(val_flat)
n_test_total = len(test_flat)

print("\nDATA SPLIT SUMMARY")
print(f"\nOriginal training data: {len(ds_train)} samples")
print(
    f"  ├── Support set (5-shot):     {n_support_total:>6} samples ({FEWSHOT_PER_CLASS} per class)"
)
print(f"  ├── Unlabeled pool:           {n_pool_total:>6} samples for pseudo-labeling")
print(f"  ├── Validation (from train):  {n_val_total:>6} samples for monitoring")
print(
    f"  └── Test (from train):        {n_test_total:>6} samples for intermediate evaluation"
)

# Multi-Backbone Feature Extractor comparison

For this project, three pre trained backbone architectures are compared to find the best one for this few shot classification task: ResNet50, EfficientNet-B4, and ViT-B/16 (vision transformer). For each model, the final classification head is removed and the last layer is used to obtain fixed length embeddings. Each backbone is tested on a dummy image to ensure that it is correctly downloaded and ready to use:

In [ ]:
print("Testing backbone + preprocessing combinations:")
for backbone_name in MultiBackboneFeatureExtractor.SUPPORTED_BACKBONES:
    for preprocess_mode in ["native"]:
        try:
            test_extractor = MultiBackboneFeatureExtractor(
                backbone_name, DEVICE, preprocess_mode
            )
            dummy_img = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
            emb = test_extractor.extract_single(dummy_img)
            print(f"  {backbone_name}: shape = {emb.shape}")
            del test_extractor
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
        except Exception as e:
            print(f"  {backbone_name}: FAILED - {e}")

print("\n All backbones ready for comparison.")

Two preprocessing methods are tested along with the three initial models:
- resizing the short image edge, center crop, and normalize
- crops to the bird's bounding box (+15% padding), then applies the backbone's native ImageNet preprocessing

In [ ]:
sample_classes = list(support_indices.keys())[:5]
viz_indices = [
    int(support_indices[c][0]) for c in sample_classes if len(support_indices[c]) > 0
]
visualize_preprocessing_modes(ds_train, viz_indices[:5])

As expected, the bounding box cropping ensures that the bird is centered in the image, and that minimal background is present. This sometimes reflects the bird, as the bird is often quite small relative to the whole image.

## Backbone comparison

For each backbone and preprocessing combination, the few-shot classification performance is evaluated by the following method:
- First, embeddings for each support image are calculated.
- The embeddings are grouped by class, and the mean embedding is calculated for each class. This is the class prototype.
- Next, all the validation images are passed through the model's frozen backbone. These are classified using cosine similarity to find the closest class prototype to the validation image.

The best performing backbone will produce the most separable embeddings between classes, and should output the highest accuracy.

In [ ]:
SKIP_BACKBONE_COMPARISON = globals().get("SKIP_BACKBONE_COMPARISON", True)
if not SKIP_BACKBONE_COMPARISON:
    print("BACKBONE + PREPROCESSING COMPARISON")
    print("Backbones: ResNet-50, EfficientNet-B4, ViT-B/16")
    print("Preprocessing: Native (center crop), Bbox Crop")

    print("\nLoading validation labels...")
    val_labels_split = get_labels_for_indices(ds_train, val_flat)

    backbone_results = []

    for backbone_name in ['resnet50', 'efficientnet_b4', 'vit_b_16']:
        for preprocess_mode in ['native', 'bbox_crop']:
            result = evaluate_backbone_fewshot(
                backbone_name=backbone_name,
                preprocess_mode=preprocess_mode,
                ds=ds_train,
                val_indices=val_flat,
                val_labels=val_labels_split,
                support_indices=support_indices,
                device=DEVICE,
                batch_size=BATCH_SIZE,
                max_val_samples=None,
                cache_dir=EMBEDDING_CACHE_DIR
            )
            backbone_results.append(result)

    print("\n" + "="*80)
    print("BACKBONE + PREPROCESSING COMPARISON SUMMARY (VALIDATION SET METRICS)")
    print("="*80)
    print(f"{'Config':<30} {'Val Accuracy':>12} {'Val F1':>10} {'Emb Dim':>10} {'Time':>10}")
    print("-"*80)

    best_result = None
    best_accuracy = 0

    for r in backbone_results:
        print(f"{r['config']:<30} {r['accuracy']*100:>11.2f}% {r['f1']*100:>9.2f}% "
              f"{r['embedding_dim']:>10} {r['time_seconds']:>9.1f}s")
        if r['accuracy'] > best_accuracy:
            best_accuracy = r['accuracy']
            best_result = r

    print("-"*80)
    print(f"\nBEST CONFIG: {best_result['config'].upper()}")
    print(f"   Validation Accuracy: {best_result['accuracy']*100:.2f}%")
    print(f"   Validation F1 Score: {best_result['f1']*100:.2f}%")

    BEST_BACKBONE = best_result['backbone']
    BEST_PREPROCESS_MODE = best_result['preprocess_mode']
else:
    print(" Skipping backbone comparison (SKIP_BACKBONE_COMPARISON=True)")
    BEST_BACKBONE = globals().get('CACHED_BEST_BACKBONE', 'resnet50')
    BEST_PREPROCESS_MODE = globals().get('CACHED_BEST_PREPROCESS_MODE', 'native')
    print(f" Using cached: {BEST_BACKBONE} + {BEST_PREPROCESS_MODE}")

In a graph:

In [ ]:
if 'backbone_results' in dir() and len(backbone_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    native_results = [r for r in backbone_results if r['preprocess_mode'] == 'native']
    bbox_results = [r for r in backbone_results if r['preprocess_mode'] == 'bbox_crop']

    backbones = [r['backbone'] for r in native_results]
    x = np.arange(len(backbones))
    width = 0.35

    # Plot 1: Validation Accuracy comparison
    ax1 = axes[0]
    bars1 = ax1.bar(x - width/2, [r['accuracy']*100 for r in native_results],
                    width, label='Native', color='#3498db', edgecolor='black')
    bars2 = ax1.bar(x + width/2, [r['accuracy']*100 for r in bbox_results],
                    width, label='Bbox Crop', color='#2ecc71', edgecolor='black')
    ax1.set_ylabel('Validation Accuracy (%)', fontsize=12)
    ax1.set_title('Validation Accuracy by Backbone', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(backbones, rotation=15)
    ax1.legend()
    ax1.set_ylim(0, max([r['accuracy']*100 for r in backbone_results]) * 1.15)

    for bar in bars1:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

    # Plot 2: Validation F1 Score comparison
    ax2 = axes[1]
    bars3 = ax2.bar(x - width/2, [r['f1']*100 for r in native_results],
                    width, label='Native', color='#3498db', edgecolor='black')
    bars4 = ax2.bar(x + width/2, [r['f1']*100 for r in bbox_results],
                    width, label='Bbox Crop', color='#2ecc71', edgecolor='black')
    ax2.set_ylabel('Validation F1 Score (%)', fontsize=12)
    ax2.set_title('Validation F1 Score by Backbone', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(backbones, rotation=15)
    ax2.legend()
    ax2.set_ylim(0, max([r['f1']*100 for r in backbone_results]) * 1.15)

    for bar in bars3:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)
    for bar in bars4:
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

    # Plot 3: Improvement from bbox crop (validation accuracy)
    ax3 = axes[2]
    improvements = []
    for i, backbone in enumerate(backbones):
        native_acc = native_results[i]['accuracy'] * 100
        bbox_acc = bbox_results[i]['accuracy'] * 100
        improvements.append(bbox_acc - native_acc)

    colors = ['#2ecc71' if imp > 0 else '#e74c3c' for imp in improvements]
    bars5 = ax3.bar(x, improvements, color=colors, edgecolor='black')
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax3.set_ylabel('Validation Accuracy Change (%)', fontsize=12)
    ax3.set_title('Bbox Crop Improvement\n(Validation Accuracy)', fontsize=14, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(backbones, rotation=15)

    for bar, imp in zip(bars5, improvements):
        y_pos = bar.get_height() + 0.08 if imp > 0 else bar.get_height() - 0.5
        ax3.text(bar.get_x() + bar.get_width()/2, y_pos,
                 f'{imp:+.1f}%', ha='center', va='bottom' if imp > 0 else 'top',
                 fontsize=10, fontweight='bold')

    plt.suptitle('Backbone + Preprocessing Comparison (Validation Set Metrics)',
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No backbone results to visualize. Run the comparison first.")

From this initial comparison, best accuracy and f1 scores are found for each model when using bounding box cropping as a preprocessing method. Moreover, the efficientnet b4 model outperformed the other two models by about 10%, and will be used for the rest of the project.

In [ ]:
CACHED_BEST_BACKBONE = "efficientnet_b4"
CACHED_BEST_PREPROCESS_MODE = "bbox_crop"

BEST_BACKBONE = CACHED_BEST_BACKBONE
BEST_PREPROCESS_MODE = CACHED_BEST_PREPROCESS_MODE

print(f"Using backbone: {BEST_BACKBONE}")
print(f"Using preprocess mode: {BEST_PREPROCESS_MODE}")


## Experiment setup

For running experiments, a feature extractor is defined:

In [ ]:
extractor = MultiBackboneFeatureExtractor(
    backbone_name=BEST_BACKBONE,
    device=DEVICE,
    preprocess_mode=BEST_PREPROCESS_MODE
)
print(f"Created extractor: {extractor.backbone_name} + {extractor.preprocess_mode}")
print(f"  Embedding dimension: {extractor.embedding_dim}")

To avoid re-extracting embeddings every time the run_learned_projection_experiment function is called, training embeddings are pre-computed:

In [ ]:
EMBEDDING_CACHE_FILE = EMBEDDING_CACHE_DIR / f"{BEST_BACKBONE}_{BEST_PREPROCESS_MODE}_all_train.npz"

if EMBEDDING_CACHE_FILE.exists():
    print("Loading pre-computed embeddings from cache...")
    cached = np.load(EMBEDDING_CACHE_FILE)
    ALL_EMBEDDINGS = cached['embeddings']
    ALL_INDICES = cached['indices']
    print(f"  Loaded {len(ALL_EMBEDDINGS)} embeddings, shape: {ALL_EMBEDDINGS.shape}")
else:
    print("Extracting all training embeddings...")
    all_indices = CLEAN_TRAIN_INDICES if CLEAN_TRAIN_INDICES is not None else np.arange(len(ds_train))
    ALL_EMBEDDINGS = extractor.extract_from_dataset(ds_train, all_indices, batch_size=BATCH_SIZE)
    ALL_INDICES = all_indices
    np.savez(EMBEDDING_CACHE_FILE, embeddings=ALL_EMBEDDINGS, indices=ALL_INDICES)
    print(f"  Saved {len(ALL_EMBEDDINGS)} embeddings to {EMBEDDING_CACHE_FILE}")
_idx_to_pos = {int(idx): pos for pos, idx in enumerate(ALL_INDICES)}

print(f"Fast embedding lookup ready. Use get_embeddings_fast(indices) for instant access.")

# Initial prototype based few-shot baseline evaluation

The best performing EfficientNet-b4 backbone is used for the full few-shot baseline experiment, and initialized with the FewShotExperiment class. This computes prototypes, predicts classes using cosine similarity, and tracks the support set and unlabeled data pool.

The FewShotExperiment is initialized, and initial results are calculated:

In [ ]:
experiment = FewShotExperiment(
    ds_train=ds_train,
    support_indices=support_indices,
    pool_indices=pool_indices,
    val_indices=val_indices_split,
    test_indices=test_indices_split,
    extractor=extractor,
    n_support=FEWSHOT_PER_CLASS,
    seed=SEED,
    cache_dir=str(EMBEDDING_CACHE_DIR),
    batch_size=BATCH_SIZE,
    use_fp16_embeddings=USE_FP16_EMBEDDINGS,
)

val_results = experiment.evaluate_on_val()
class_ids = experiment.class_ids

baseline_metrics = {
    "accuracy": val_results["accuracy"],
    "precision": val_results["precision"],
    "recall": val_results["recall"],
    "f1": val_results["f1"],
    "n_samples": len(experiment.val_flat),
}

print(f"Model: {extractor.backbone_name} + {extractor.preprocess_mode}")
print(f"Embedding dim: {extractor.embedding_dim} | Classes: {len(class_ids)}\n")

baseline_df = pd.DataFrame(
    {
        "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
        "Value": [
            f"{baseline_metrics['accuracy']*100:.2f}%",
            f"{baseline_metrics['precision']*100:.2f}%",
            f"{baseline_metrics['recall']*100:.2f}%",
            f"{baseline_metrics['f1']*100:.2f}%",
        ],
    }
).set_index("Metric")

display(baseline_df)

initial_accuracy = baseline_metrics["accuracy"]
print(f"\nValidation samples: {baseline_metrics['n_samples']}")
print(f"Random baseline: {100/len(class_ids):.2f}%")

# Learned projection baseline

The original ImageNet training data likely did not use North American bird species, some of which are highly similar to each other. The UMAP plot in the eda.ipynb notebook also showed that similar embeddings for species such as hummingbirds and chickadees, as different types of subspecies often have similar shapes, colorings, poses, and feathers.

To increase validation accuracy, the learned projection approach is used, where a trainable projection head is be added to the model to optimize the generic ImageNet embeddings for this fine-grained bird classificaiton problem. After training, prototype matching is still used to classify new samples with cosine similarity.

In [ ]:
results = run_learned_projection_experiment(
    extractor=extractor,
    ds_train=ds_train,
    support_indices=support_indices,
    pool_indices=pool_indices,
    val_indices=val_indices_split,
    device=DEVICE,
    n_epochs=50,
    batch_size=64,
    lr=1e-3,
    embedding_dim=512,
    use_pool_data=False,
    seed=SEED,
    embedding_lookup_fn=get_embeddings_fast,
)

Using the learned projection method has increased validation accuracy by almost 10%, though significant overfitting is seen due to the limited training data. Now, high confidence samples from the unlabeled sample pool can be pseudo-labeled with this model to further attempt to increase accuracy. Applying the learned projection to the few shot experiment:

In [ ]:
experiment_lp = FewShotExperiment(
    ds_train=ds_train,
    support_indices={k: v.copy() for k, v in support_indices.items()},
    pool_indices={k: v.copy() for k, v in pool_indices.items()},
    val_indices=val_indices_split,
    test_indices=test_indices_split,
    extractor=extractor,
    n_support=FEWSHOT_PER_CLASS,
    seed=SEED,
    cache_dir=str(EMBEDDING_CACHE_DIR),
    batch_size=BATCH_SIZE,
    use_fp16_embeddings=USE_FP16_EMBEDDINGS,
)
experiment_lp.set_projection_head(results["model"], device=DEVICE)
baseline_acc = experiment_lp.evaluate_on_val()["accuracy"]
print(f"Experiment created with learned projection")
print(f"   Validation accuracy: {baseline_acc*100:.2f}%")
print(f"   Support size: {experiment_lp.get_support_count()}")
print(f"   Pool size: {experiment_lp.get_pool_count()}")

# Iterative pseudo-labeling

Now that a baseline has been created and a learned projection applied, iterative pseudo labeling is used to expand the training data. To do this,
- Samples from the unlabeled pool are classified using prototype distances.
- The classified pool samples with the highest confidence are selected and added to the training set.
- Next, the prototypes are recomputed with the expanded training data, and the process repeats.

If no samples have confidence higher than the threshold, this is lowered to continue expanding the training data. This loop stops when either the target validation accuracy (70%) is obtained, when no candidates are found for 3 consecutive iterations, no samples are left, or when the maximum iterations have been reached.

This is an automatic process, because manual pseudo-labeling method would be quite inefficient. Those who are not experts in North American bird species will likely not be able to correctly classify closely related species. Moreover, manually examining many images is impractical, as there are around 14 000 samples in the pool.

In [ ]:
state_before = experiment_lp.save_state()
accuracy_before = experiment_lp.evaluate_on_val()["accuracy"]
support_before = experiment_lp.get_support_count()

print(f"Starting state saved for potential rollback")
print(f"  Validation accuracy: {accuracy_before*100:.2f}%")
print(f"  Support samples:     {support_before}\n")
pl_results = experiment_lp.run_auto_pseudo_labeling(
    target_accuracy=0.70,
    initial_threshold=0.60,
    min_threshold=0.30,
    threshold_decay=0.05,
    max_iterations=15,
    max_per_class=5,
    use_true_labels=False,
    verbose=True,
)
samples_added = experiment_lp.get_support_count() - support_before
accuracy_after = experiment_lp.evaluate_on_val()["accuracy"]

print(f"\n{'='*60}")
print(f"PSEUDO-LABELING ROUND 1 COMPLETE")
print(f"{'='*60}")
print(f"  Samples added:          {samples_added}")
print(
    f"  Validation accuracy:    {accuracy_before*100:.2f}% → {accuracy_after*100:.2f}%"
)
print(f"  Change:                 {(accuracy_after - accuracy_before)*100:+.2f}%")
print(f"{'='*60}")

The first round of pseudo labeling samples increased validation accuracy minimally (0.03%) with a little over 3000 samples added to the training data. To ensure that samples are mostly correctly labeled, a samples of these added samples are visualized and carefully compared with a sample image of their labeled class to decide whether to accept or reject the new samples. As an extra check, the real class is also tracked:

In [ ]:
spot_results = spot_check_pseudo_labels(experiment_lp, n_samples=20, seed=42)
if spot_results and "error_rate" in spot_results:
    print(f"\nSpot-check error rate: {spot_results['error_rate']*100:.1f}%")
    if spot_results["error_rate"] > 0.15:
        print("High error rate - consider rollback")
    else:
        print("Acceptable error rate")

Out of this 20 class sample, only 2 of them were incorrectly labeled, and this batch of labeled samples is added to the training data:

In [ ]:
KEEP_CHANGES = True

if KEEP_CHANGES:
    print("Keeping pseudo-labeling changes")
    print(f"   Current support size: {experiment_lp.get_support_count()} samples")
else:
    experiment_lp.restore_state(state_before)
    print("Rolled back to previous state")
    print(f"   Restored support size: {experiment_lp.get_support_count()} samples")

As quite a few samples were added (even though accuracy has not increased), the projection head is retrained so that the model can learn from the larger training data:

In [ ]:
support_indices_expanded = experiment_lp.support_indices.copy()
n_expanded = sum(len(v) for v in support_indices_expanded.values())
print(f"Training on expanded support set: {n_expanded} samples")
results_retrained = run_learned_projection_experiment(
    extractor=extractor,
    ds_train=ds_train,
    support_indices=support_indices_expanded,
    pool_indices=experiment_lp.pool_indices,
    val_indices=val_indices_split,
    device=DEVICE,
    use_pool_data=False,
    batch_size=64,
    n_epochs=50,
    lr=1e-3,
    embedding_dim=512,
    seed=SEED,
    embedding_lookup_fn=get_embeddings_fast,
)

The retrained model is applied to the experiment:

In [ ]:
experiment_lp.set_projection_head(results_retrained["model"], device=DEVICE)

new_acc = experiment_lp.evaluate_on_val()["accuracy"]
print(f"\n{'='*60}")
print(f"ROUND 1 COMPLETE")
print(f"{'='*60}")
print(f"  Initial accuracy:  {baseline_acc*100:.2f}%")
print(f"  After pseudo-label: {accuracy_after*100:.2f}%")
print(f"  After retraining:   {new_acc*100:.2f}%")
print(f"  Total improvement:  {(new_acc - baseline_acc)*100:+.2f}%")
print(f"{'='*60}")

The first round of pseudo labeling increased validation accuracy by over 3%. Continuing with the same steps as before:

## Continuing pseudo-labeling

Adding more samples to the support set:

In [ ]:
round_num = 7

state_before = experiment_lp.save_state()
accuracy_before = experiment_lp.evaluate_on_val()["accuracy"]
support_before = experiment_lp.get_support_count()

print(f"{'='*60}")
print(f"ROUND {round_num} - PSEUDO-LABELING")
print(f"{'='*60}")
print(f"  Starting accuracy: {accuracy_before*100:.2f}%")
print(f"  Starting support:  {support_before}")

pl_results = experiment_lp.run_auto_pseudo_labeling(
    target_accuracy=0.80,
    initial_threshold=0.4,
    min_threshold=0.25,
    threshold_decay=0.05,
    max_iterations=10,
    max_per_class=5,
    use_true_labels=False,
    verbose=True,
)

samples_added = experiment_lp.get_support_count() - support_before
accuracy_after = experiment_lp.evaluate_on_val()["accuracy"]

print(f"\n{'='*60}")
print(f"ROUND {round_num} PSEUDO-LABELING COMPLETE")
print(f"{'='*60}")
print(f"  Samples added:       {samples_added}")
print(f"  Accuracy:            {accuracy_before*100:.2f}% → {accuracy_after*100:.2f}%")
print(f"  Change:              {(accuracy_after - accuracy_before)*100:+.2f}%")

Looking at some newly classified samples:

In [ ]:
spot_results = spot_check_pseudo_labels(experiment_lp, n_samples=20, seed=round_num)

if spot_results and "error_rate" in spot_results:
    print(f"\nSpot-check error rate: {spot_results['error_rate']*100:.1f}%")
    if spot_results["error_rate"] > 0.20:
        print("High error rate - consider rollback")
    else:
        print("Acceptable error rate")

Only one out of 20 samples was misclassified, so this batch of samples is added to the training set:

In [ ]:
KEEP_CHANGES = True

if KEEP_CHANGES:
    print(f"Keeping Round {round_num} changes")
    print(f"   Support size: {experiment_lp.get_support_count()}")
else:
    experiment_lp.restore_state(state_before)
    print(f"Rolled back Round {round_num}")
    print(f"   Support size: {experiment_lp.get_support_count()}")

As about 2000 samples were added to the training data set for the second pseudo-labeling round, the projection head is retrained again:

In [ ]:
n_expanded = sum(len(v) for v in experiment_lp.support_indices.values())
print(f"Retraining on {n_expanded} samples...")

results_retrained = run_learned_projection_experiment(
    extractor=extractor,
    ds_train=ds_train,
    support_indices=experiment_lp.support_indices.copy(),
    pool_indices=experiment_lp.pool_indices,
    val_indices=val_indices_split,
    device=DEVICE,
    use_pool_data=False,
    batch_size=64,
    n_epochs=50,
    lr=1e-3,
    embedding_dim=512,
    seed=SEED,
    embedding_lookup_fn=get_embeddings_fast,
)

Validation accuracy has now increased to 68%. Now, the retrained model is applied to the experiment:

In [ ]:
experiment_lp.set_projection_head(results_retrained["model"], device=DEVICE)

new_acc = experiment_lp.evaluate_on_val()["accuracy"]

print(f"\n{'='*60}")
print(f"ROUND {round_num} COMPLETE")
print(f"{'='*60}")
print(f"  Accuracy after retrain: {new_acc*100:.2f}%")
print(f"  Support size:           {experiment_lp.get_support_count()}")
if new_acc >= 0.70:
    print(f"TARGET REACHED!")
else:
    print(f"\nGap to 70%: {(0.70 - new_acc)*100:.2f}%")
    print("Run more rounds if needed")

As accuracy has not reached 70%, this loop is re-run. Note: if less than 500 samples are added to the training set, then the pseudo-labeling is repeated before retraining the projection head.

## Pseudo labeling history:

After all rounds of pseudo-labeling, validation accuracy has reached 78%, which is just above the target for this task. Examining the pseudo-labeling progress:

In [ ]:
history = experiment_lp.history

if history:
    records = []
    for h in history:
        records.append(
            {
                "iteration": h.get("iteration", len(records) + 1),
                "samples_added": h.get("samples_added", h.get("n_added", 0)),
                "val_accuracy": h.get("accuracy_after", h.get("val_accuracy", 0)),
                "support_size": h.get("support_count", h.get("support_size", 0)),
            }
        )

    df = pd.DataFrame(records)

    print("PSEUDO-LABELING HISTORY")
    print("=" * 60)
    print(df.to_string(index=False))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(
        df["iteration"], df["val_accuracy"] * 100, "b-o", linewidth=2, markersize=6
    )
    axes[0].axhline(
        y=70, color="green", linestyle="--", linewidth=2, label="Target (70%)"
    )
    axes[0].fill_between(df["iteration"], df["val_accuracy"] * 100, alpha=0.3)
    axes[0].set_xlabel("Iteration")
    axes[0].set_ylabel("Validation Accuracy (%)")
    axes[0].set_title("Validation Accuracy Progression")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].bar(df["iteration"], df["samples_added"], color="steelblue", alpha=0.7)
    axes[1].set_xlabel("Iteration")
    axes[1].set_ylabel("Samples Added")
    axes[1].set_title("Pseudo-Labeled Samples Added per Iteration")
    axes[1].grid(True, alpha=0.3, axis="y")

    plt.suptitle(
        "Learned Projection + Pseudo-Labeling Progress", fontsize=14, fontweight="bold"
    )
    plt.tight_layout()
    plt.show()
else:
    print("No pseudo-labeling history available yet. Run the iteration loop first.")
final_acc = experiment_lp.evaluate_on_val()["accuracy"]

print(f"\n{'='*60}")
print(f"FINAL RESULTS")
print(f"{'='*60}")
print(f"   Final validation accuracy:  {final_acc*100:.2f}%")
print(f"   Final support size:         {experiment_lp.get_support_count()}")
print(f"   Pool remaining:             {experiment_lp.get_pool_count()}")
print(f"   Target reached:             {'YES' if final_acc >= 0.70 else 'NO'}")
print(f"{'='*60}")

As the training data grows, the model is able to generalize better and increase classification accuracy over 555 classes. The final model can now be evaluated with the reserved test data (15% of the initial training data)

# Model evaluation

Now that the learned projection model has reached validation accuracy of 74% after pseudo labeling, it is evaluated on the test data split from the initial training data.

In [ ]:
print("=" * 80)
print("LEARNED PROJECTION EVALUATION ON 15% TEST SPLIT")
print("=" * 80)
try:
    class_names = ds_train.labels.info.class_names
except:
    class_names = [f"Class_{i}" for i in range(555)]

required_vars = [
    "experiment_lp",
    "test_indices_split",
    "extractor",
    "ds_train",
    "DEVICE",
]
missing = [v for v in required_vars if v not in dir() and v not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required variables: {missing}. Run the Learned Projection section first."
    )

support_indices_expanded = experiment_lp.support_indices
support_flat, support_labels = flatten_class_indices(support_indices_expanded)
print(
    f"Support set (after pseudo-labeling): {len(support_flat)} samples across {len(support_indices_expanded)} classes"
)
test_flat, test_labels_from_dict = flatten_class_indices(test_indices_split)
print(
    f"Test set (15% split): {len(test_flat)} samples across {len(test_indices_split)} classes"
)
if (
    hasattr(experiment_lp, "_projection_model")
    and experiment_lp._projection_model is not None
):
    model = experiment_lp._projection_model
    print(f"Model: {model.__class__.__name__} loaded from experiment_lp")
elif "results_retrained" in dir() or "results_retrained" in globals():
    model = results_retrained["model"]
    print(f"Model: {model.__class__.__name__} loaded from results_retrained")
else:
    raise RuntimeError("No trained model found! Run set_projection_head() first.")

model.eval()
model.to(DEVICE)
print("\nStep 1/3: Computing prototypes from expanded support set...")
support_indices_array = np.array(support_flat, dtype=np.int64)
support_embeddings_raw = get_embeddings_fast(support_indices_array)

with torch.no_grad():
    support_embeddings_tensor = (
        torch.from_numpy(support_embeddings_raw).float().to(DEVICE)
    )
    if hasattr(model, "get_embedding"):
        support_embeddings_proj = model.get_embedding(support_embeddings_tensor)
    else:
        support_embeddings_proj = model(support_embeddings_tensor)
    support_embeddings_proj = support_embeddings_proj.cpu().numpy()

prototypes = {}
for i, (idx, label) in enumerate(zip(support_flat, support_labels)):
    if label not in prototypes:
        prototypes[label] = []
    prototypes[label].append(support_embeddings_proj[i])

for label in prototypes:
    prototypes[label] = np.mean(prototypes[label], axis=0)

print(f"  Created prototypes for {len(prototypes)} classes")
print("\nStep 2/3: Extracting test embeddings...")

test_indices_array = np.array(test_flat, dtype=np.int64)
test_embeddings_raw = get_embeddings_fast(test_indices_array)

with torch.no_grad():
    test_embeddings_tensor = torch.from_numpy(test_embeddings_raw).float().to(DEVICE)
    if hasattr(model, "get_embedding"):
        test_embeddings_proj = model.get_embedding(test_embeddings_tensor)
    else:
        test_embeddings_proj = model(test_embeddings_tensor)
    test_embeddings_proj = test_embeddings_proj.cpu().numpy()

test_true_labels = test_labels_from_dict

print("\nStep 3/3: Computing predictions...")

prototype_labels = sorted(prototypes.keys())
prototype_matrix = np.stack([prototypes[lbl] for lbl in prototype_labels])
test_embeddings_norm = test_embeddings_proj / (
    np.linalg.norm(test_embeddings_proj, axis=1, keepdims=True) + 1e-8
)
prototype_matrix_norm = prototype_matrix / (
    np.linalg.norm(prototype_matrix, axis=1, keepdims=True) + 1e-8
)
similarities = test_embeddings_norm @ prototype_matrix_norm.T
best_proto_idx = np.argmax(similarities, axis=1)
test_predictions = np.array([prototype_labels[i] for i in best_proto_idx])
test_confidences = np.array(
    [similarities[i, best_proto_idx[i]] for i in range(len(test_flat))]
)
test_true_labels = np.array(test_true_labels)

correct = (test_predictions == test_true_labels).sum()
accuracy = correct / len(test_true_labels)
precision = precision_score(
    test_true_labels, test_predictions, average="macro", zero_division=0
)
recall = recall_score(
    test_true_labels, test_predictions, average="macro", zero_division=0
)
f1 = f1_score(test_true_labels, test_predictions, average="macro", zero_division=0)

print("\n" + "=" * 80)
print(f"TEST SPLIT RESULTS (15% held-out from training data)")
print("=" * 80)
print(f"  Accuracy:  {accuracy:.2%} ({correct}/{len(test_true_labels)})")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print("=" * 80)

class_correct = {}
class_total = {}
class_sample_indices = {}

for i, (true_label, pred_label) in enumerate(zip(test_true_labels, test_predictions)):
    true_label = int(true_label)
    if true_label not in class_total:
        class_total[true_label] = 0
        class_correct[true_label] = 0
        class_sample_indices[true_label] = []
    class_total[true_label] += 1
    class_sample_indices[true_label].append(i)
    if true_label == pred_label:
        class_correct[true_label] += 1

class_accuracies = {c: class_correct[c] / class_total[c] for c in class_total}
sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1])
worst_5 = sorted_classes[:5]
best_5 = sorted_classes[-5:][::-1]

print("\n" + "-" * 80)
print("WORST PERFORMING CLASSES")
print("-" * 80)
for class_id, acc in worst_5:
    species_name = (
        class_names[class_id] if class_id < len(class_names) else f"Unknown_{class_id}"
    )
    print(f"  Class {class_id}: {species_name[:40]}")
    print(
        f"    Accuracy: {acc:.1%} ({class_correct[class_id]}/{class_total[class_id]})"
    )

print("\n" + "-" * 80)
print("BEST PERFORMING CLASSES")
print("-" * 80)
for class_id, acc in best_5:
    species_name = (
        class_names[class_id] if class_id < len(class_names) else f"Unknown_{class_id}"
    )
    print(f"  Class {class_id}: {species_name[:40]}")
    print(
        f"    Accuracy: {acc:.1%} ({class_correct[class_id]}/{class_total[class_id]})"
    )
all_accuracies = list(class_accuracies.values())
zero_acc_classes = sum(1 for a in all_accuracies if a == 0)
perfect_acc_classes = sum(1 for a in all_accuracies if a == 1.0)

print("\n" + "-" * 80)
print("SUMMARY STATISTICS")
print("-" * 80)
print(f"  Overall Accuracy:           {accuracy:.2%}")
print(f"  Mean Per-Class Accuracy:    {np.mean(all_accuracies):.2%}")
print(f"  Median Per-Class Accuracy:  {np.median(all_accuracies):.2%}")
print(f"  Std Per-Class Accuracy:     {np.std(all_accuracies):.2%}")
print(f"  Classes with 0% accuracy:   {zero_acc_classes}")
print(f"  Classes with 100% accuracy: {perfect_acc_classes}")
print(f"  Support set size:           {len(support_flat)}")
print("=" * 80)

The first test data also shows 73% accuracy with slightly lower recall (71%) and F1 score (69%). Some classes such as the rchard Oriole or the Blue-winged Teal have 0% accuracy, and classes such as the Surfbird or the Pacific Galunule have 100% accuracy. This is an indication that the model is separating some classes well, but is still getting easily confused with other species.

As initial test predictions have met the task objective (70% accuracy), the final holdout test set is loaded:

In [ ]:
print("=" * 70)
print("LOADING FINAL HOLDOUT TEST SET (ds_val)")
print("=" * 70)

ds_val = deeplake.load("hub://activeloop/nabirds-dataset-val", read_only=True)
print(f"Holdout samples (raw): {len(ds_val)}")
CLEAN_VAL_INDICES_PATHS = [
    Path(MODULE_PATH) / "clean_indices.npz",
    Path("clean_indices.npz"),
    DATA_ROOT / "clean_indices.npz",
]
_clean_val_npz = None
for p in CLEAN_VAL_INDICES_PATHS:
    if p.exists():
        _clean_val_npz = np.load(p)
        print(f"Loaded cleaned index file: {p}")
        break

CLEAN_VAL_INDICES = None
if _clean_val_npz is not None and "val_indices" in _clean_val_npz.files:
    CLEAN_VAL_INDICES = _clean_val_npz["val_indices"].astype(np.int64)
    print(f"Using cleaned val indices (duplicates dropped): {len(CLEAN_VAL_INDICES)} kept")

VAL_LABEL_INDEX_CANDIDATES = [
    Path(MODULE_PATH) / "label_index_val_clean.npz",
    Path("label_index_val_clean.npz"),
    DATA_ROOT / "label_index_val_clean.npz",
    Path(MODULE_PATH) / "label_index_val.npz",
    Path("label_index_val.npz"),
    DATA_ROOT / "label_index_val.npz",
]
VAL_LABEL_INDEX_PATH = next((p for p in VAL_LABEL_INDEX_CANDIDATES if p.exists()), None)

if VAL_LABEL_INDEX_PATH is not None:
    val_label_index = load_label_index(VAL_LABEL_INDEX_PATH)
    print(f"Loaded validation label index from {VAL_LABEL_INDEX_PATH}")
else:
    val_label_index = build_label_index(ds_val)
    if CLEAN_VAL_INDICES is not None:
        allowed = set(map(int, CLEAN_VAL_INDICES))
        val_label_index = {
            int(class_id): np.array([int(i) for i in indices if int(i) in allowed], dtype=np.int64)
            for class_id, indices in val_label_index.items()
        }
    out_path = DATA_ROOT / "label_index_val_clean.npz"
    save_label_index(val_label_index, out_path)
    print(f"Built and saved validation label index to {out_path}")

final_test_indices = []
final_test_labels = []
for class_id, indices in val_label_index.items():
    final_test_indices.extend(indices.tolist())
    final_test_labels.extend([class_id] * len(indices))

final_test_indices = np.array(final_test_indices, dtype=np.int64)
final_test_labels = np.array(final_test_labels, dtype=np.int64)

val_num_classes = len(val_label_index)
print(f"Final holdout test set prepared:")
print(f"   Total samples: {len(final_test_labels)}")
print(f"   Unique classes: {val_num_classes}")

Now, the final holdout test data is evaluated:

In [ ]:
print("=" * 80)
print("HOLDOUT EVALUATION (ds_val) WITH LEARNED PROJECTION")
print("=" * 80)
required_vars = [
    "experiment_lp",
    "ds_val",
    "final_test_indices",
    "final_test_labels",
    "extractor",
    "DEVICE",
    "HOLDOUT_CACHE_DIR",
]
missing = [v for v in required_vars if v not in dir() and v not in globals()]
if missing:
    raise RuntimeError(f"Missing required variables: {missing}")
try:
    class_names = ds_train.labels.info.class_names
except:
    class_names = [f"Class_{i}" for i in range(555)]

# === STEP 1: Get trained model and compute prototypes from support set ===
print("\nStep 1/3: Loading model and computing prototypes from support set...")

if (
    hasattr(experiment_lp, "_projection_model")
    and experiment_lp._projection_model is not None
):
    model = experiment_lp._projection_model
    print(f"  Model: {model.__class__.__name__}")
elif "results_retrained" in dir() or "results_retrained" in globals():
    model = results_retrained["model"]
    print(f"  Model: {model.__class__.__name__}")
else:
    raise RuntimeError("No trained model found! Run set_projection_head() first.")

model.eval()
model.to(DEVICE)

# Get support set from experiment (uses training data - cached embeddings)
support_indices_expanded = experiment_lp.support_indices
support_flat, support_labels = flatten_class_indices(support_indices_expanded)
print(
    f"  Support set: {len(support_flat)} samples across {len(support_indices_expanded)} classes"
)

# Compute prototypes using cached training embeddings
support_indices_array = np.array(support_flat, dtype=np.int64)
support_embeddings_raw = get_embeddings_fast(support_indices_array)

with torch.no_grad():
    support_embeddings_tensor = (
        torch.from_numpy(support_embeddings_raw).float().to(DEVICE)
    )
    if hasattr(model, "get_embedding"):
        support_embeddings_proj = model.get_embedding(support_embeddings_tensor)
    else:
        support_embeddings_proj = model(support_embeddings_tensor)
    support_embeddings_proj = support_embeddings_proj.cpu().numpy()

# Average embeddings per class to get prototypes
prototypes = {}
for i, label in enumerate(support_labels):
    if label not in prototypes:
        prototypes[label] = []
    prototypes[label].append(support_embeddings_proj[i])

for label in prototypes:
    prototypes[label] = np.mean(prototypes[label], axis=0)

print(f"  Created prototypes for {len(prototypes)} classes")

# === STEP 2: Extract embeddings from holdout set (ds_val) ===
print("\nStep 2/3: Extracting holdout embeddings from ds_val...")
print(f"  Holdout samples: {len(final_test_indices)}")
holdout_embeddings_raw = extractor.extract_from_dataset(
    ds_val, final_test_indices, batch_size=64, show_progress=True
)
holdout_true_labels = np.array(final_test_labels)

print(f"  Extracted {len(holdout_embeddings_raw)} embeddings")

# === STEP 3: Project holdout embeddings and classify ===
print("\nStep 3/3: Projecting and classifying...")

with torch.no_grad():
    holdout_tensor = torch.from_numpy(holdout_embeddings_raw).float().to(DEVICE)
    if hasattr(model, "get_embedding"):
        holdout_embeddings_proj = model.get_embedding(holdout_tensor)
    else:
        holdout_embeddings_proj = model(holdout_tensor)
    holdout_embeddings_proj = holdout_embeddings_proj.cpu().numpy()

# Compute predictions via cosine similarity to prototypes
prototype_labels = sorted(prototypes.keys())
prototype_matrix = np.stack([prototypes[lbl] for lbl in prototype_labels])

holdout_norm = holdout_embeddings_proj / (
    np.linalg.norm(holdout_embeddings_proj, axis=1, keepdims=True) + 1e-8
)
prototype_norm = prototype_matrix / (
    np.linalg.norm(prototype_matrix, axis=1, keepdims=True) + 1e-8
)

similarities = holdout_norm @ prototype_norm.T
best_proto_idx = np.argmax(similarities, axis=1)
holdout_predictions = np.array([prototype_labels[i] for i in best_proto_idx])
holdout_confidences = np.array(
    [similarities[i, best_proto_idx[i]] for i in range(len(holdout_true_labels))]
)

# === RESULTS ===
correct = (holdout_predictions == holdout_true_labels).sum()
accuracy = correct / len(holdout_true_labels)
precision = precision_score(
    holdout_true_labels, holdout_predictions, average="macro", zero_division=0
)
recall = recall_score(
    holdout_true_labels, holdout_predictions, average="macro", zero_division=0
)
f1 = f1_score(
    holdout_true_labels, holdout_predictions, average="macro", zero_division=0
)

print("\n" + "=" * 80)
print("HOLDOUT RESULTS (ds_val - completely unseen during training)")
print("=" * 80)
print(f"  Accuracy:  {accuracy:.2%} ({correct}/{len(holdout_true_labels)})")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print("=" * 80)
class_correct = {}
class_total = {}
for true_label, pred_label in zip(holdout_true_labels, holdout_predictions):
    true_label = int(true_label)
    class_total[true_label] = class_total.get(true_label, 0) + 1
    class_correct[true_label] = class_correct.get(true_label, 0) + (
        1 if true_label == pred_label else 0
    )

class_accuracies = {c: class_correct[c] / class_total[c] for c in class_total}
sorted_classes = sorted(class_accuracies.items(), key=lambda x: x[1])

print("\n" + "-" * 80)
print("WORST 5 CLASSES ON HOLDOUT")
print("-" * 80)
for class_id, acc in sorted_classes[:5]:
    species_name = (
        class_names[class_id] if class_id < len(class_names) else f"Unknown_{class_id}"
    )
    print(
        f"  {class_id}: {species_name[:40]} - {acc:.1%} ({class_correct[class_id]}/{class_total[class_id]})"
    )

print("\n" + "-" * 80)
print("BEST 5 CLASSES ON HOLDOUT")
print("-" * 80)
for class_id, acc in sorted_classes[-5:][::-1]:
    species_name = (
        class_names[class_id] if class_id < len(class_names) else f"Unknown_{class_id}"
    )
    print(
        f"  {class_id}: {species_name[:40]} - {acc:.1%} ({class_correct[class_id]}/{class_total[class_id]})"
    )

all_accs = list(class_accuracies.values())
print("\n" + "-" * 80)
print("HOLDOUT SUMMARY")
print("-" * 80)
print(f"  Overall Accuracy:          {accuracy:.2%}")
print(f"  Mean Per-Class Accuracy:   {np.mean(all_accs):.2%}")
print(f"  Median Per-Class Accuracy: {np.median(all_accs):.2%}")
print(f"  Classes with 0% accuracy:  {sum(1 for a in all_accs if a == 0)}")
print(f"  Classes with 100% accuracy:{sum(1 for a in all_accs if a == 1.0)}")
print("=" * 80)

Final test accuracy (71.69%) is slightly lower than that of the validation and test data split from the initial training set, though it does meet the taks objective of 70%. F1 score is about 3% lower than accuracy, which is an indication that the model is good at predicting majority classes, but struggles with minority classes. No classes have 0% accuracy, though only 13 classes have accuracy of 100%. This is indicative that the model is generalizing fairly well, but is making quite a few mistakes.

Examining the best and worst performing classes:

In [ ]:
holdout_class_stats = defaultdict(lambda: {"correct": 0, "total": 0, "predictions": []})

for true_label, pred_label in zip(holdout_true_labels, holdout_predictions):
    true_label = int(true_label)
    pred_label = int(pred_label)
    holdout_class_stats[true_label]["total"] += 1
    holdout_class_stats[true_label]["predictions"].append(pred_label)
    if true_label == pred_label:
        holdout_class_stats[true_label]["correct"] += 1

for class_id in holdout_class_stats:
    stats = holdout_class_stats[class_id]
    stats["accuracy"] = stats["correct"] / stats["total"] if stats["total"] > 0 else 0.0
    stats["support_size"] = stats["total"]

try:
    class_names = ds_val.labels.info.class_names
except:
    try:
        class_names = ds_train.labels.info.class_names
    except:
        class_names = [f"Class_{i}" for i in range(NUM_CLASSES)]

holdout_accuracies = np.array(
    [holdout_class_stats[c]["accuracy"] for c in holdout_class_stats.keys()]
)

print("=" * 70)
print("PER-CLASS ACCURACY STATISTICS (HOLDOUT)")
print("=" * 70)
print(f"Total classes evaluated: {len(holdout_class_stats)}")
print(f"Mean per-class accuracy: {np.mean(holdout_accuracies)*100:.2f}%")
print(f"Median per-class accuracy: {np.median(holdout_accuracies)*100:.2f}%")
print(f"Std per-class accuracy: {np.std(holdout_accuracies)*100:.2f}%")
classes_above_70 = sum(1 for acc in holdout_accuracies if acc >= 0.70)
print(
    f"\nClasses ≥70% accuracy: {classes_above_70}/{len(holdout_accuracies)} ({classes_above_70/len(holdout_accuracies)*100:.1f}%)"
)


holdout_sample_info = []
for i, (true_lbl, pred_lbl) in enumerate(zip(holdout_true_labels, holdout_predictions)):
    holdout_sample_info.append(
        {
            "idx": i,
            "ds_idx": int(final_test_indices[i]),
            "true": int(true_lbl),
            "pred": int(pred_lbl),
            "correct": int(true_lbl) == int(pred_lbl),
        }
    )

worst_5 = sorted(holdout_class_stats.items(), key=lambda x: x[1]["accuracy"])[:5]
best_5 = sorted(
    holdout_class_stats.items(), key=lambda x: x[1]["accuracy"], reverse=True
)[:5]


print("=" * 70)
print("WORST PERFORMING CLASSES - Sample Images")
print("=" * 70)
show_class_samples(
    worst_5, "Worst 5 Classes (Showing Misclassifications)", show_mistakes=True
)

print("\n" + "=" * 70)
print("BEST PERFORMING CLASSES - Sample Images")
print("=" * 70)
show_class_samples(best_5, "Best 5 Classes (100% Accuracy)", show_mistakes=False)

Over half of the classes have over 70% accuracy, which indicates that the model is performing relatively well. The class accuracy standard deviation is quite large, which indicates that some classes are quite accurate, and some classes are badly recognized by the model. Classes with very low accuracy have examples that often have different backgrounds, lighting conditions, and bird poses. Also, some of these misclassified examples were predicted as another subspecies (for example, the Chihuahuan Raven was predicted as a Common Raven). Classes with the highest accuracy generally have the same bird position in the picture, with the bird generally showing a side profile and not flying.

Examining the class accuracy distribution:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 1. Histogram of per-class accuracies
ax1 = axes[0]
ax1.hist(holdout_accuracies, bins=25, edgecolor="black", alpha=0.7, color="steelblue")
ax1.axvline(
    np.mean(holdout_accuracies),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(holdout_accuracies)*100:.1f}%",
)
ax1.axvline(
    np.median(holdout_accuracies),
    color="orange",
    linestyle="--",
    linewidth=2,
    label=f"Median: {np.median(holdout_accuracies)*100:.1f}%",
)
ax1.axvline(0.7, color="green", linestyle=":", linewidth=2, label="Target: 70%")
ax1.set_xlabel("Per-Class Accuracy", fontsize=12)
ax1.set_ylabel("Number of Classes", fontsize=12)
ax1.set_title("Distribution of Per-Class Accuracy (Holdout)", fontsize=14)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Accuracy vs Support Size scatter
ax2 = axes[1]
support_sizes = [
    holdout_class_stats[c]["support_size"] for c in holdout_class_stats.keys()
]
class_accs = [holdout_class_stats[c]["accuracy"] for c in holdout_class_stats.keys()]
ax2.scatter(support_sizes, class_accs, alpha=0.5, c="steelblue", edgecolor="white")
ax2.set_xlabel("Support Set Size (samples per class)", fontsize=12)
ax2.set_ylabel("Per-Class Accuracy", fontsize=12)
ax2.set_title("Accuracy vs Support Size (Holdout)", fontsize=14)
ax2.axhline(0.7, color="green", linestyle="--", alpha=0.7, label="Target 70%")
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.suptitle(
    "Holdout Test Set: Per-Class Performance Analysis",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

The distribution of per-class accuracy shows that most classes have relatively high accuracy (over 70%), though there are less than 60 classes with a perfect accuracy. No clear trend is seen when examining accuracy vs support set size, which shows that the model generalizes fairly well to minority classes. Examining the confusion matrix for the top 15 most confused pairs:


In [ ]:
n_pairs_to_show = 10
pair_classes = []
for pair in confused_pairs[:n_pairs_to_show]:
    pair_classes.append((pair["true_class"], pair["pred_class"]))
unique_classes = sorted(set(c for pair in pair_classes for c in pair))
valid_classes = [c for c in unique_classes if c in holdout_class_ids]
n_cls = len(valid_classes)
cm_focused = np.zeros((n_cls, n_cls), dtype=int)
class_to_idx = {c: i for i, c in enumerate(valid_classes)}

for i, (true_lbl, pred_lbl) in enumerate(
    zip(holdout_labels_array, holdout_preds_array)
):
    true_lbl, pred_lbl = int(true_lbl), int(pred_lbl)
    if true_lbl in class_to_idx and pred_lbl in class_to_idx:
        cm_focused[class_to_idx[true_lbl], class_to_idx[pred_lbl]] += 1

short_names = []
for c in valid_classes:
    name = class_names[c] if c < len(class_names) else f"Class_{c}"
    short_name = name.split("/")[-1] if "/" in name else name
    short_names.append(short_name[:28])
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_focused,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=short_names,
    yticklabels=short_names,
    cbar_kws={"label": "Count"},
)
plt.xlabel("Predicted Species", fontsize=12)
plt.ylabel("True Species", fontsize=12)
plt.title(
    f"Top {n_pairs_to_show} Most Confused Pairs",
    fontsize=14,
)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In general, one species is not frequently confused as multiple other species. A general trend of confusing subspecies is seen. For example, Cooper’s Hawk is often misclassified as the Sharp-shinned Hawk, and two species of Norther Flicker are often confused with each other. Visually examining some of these confused pairs:

In [ ]:
print("=" * 100)
print("MOST CONFUSED CLASS PAIRS - Visual Comparison (Holdout)")
print("=" * 100)

for pair_idx, pair in enumerate(confused_pairs[:5]):
    true_cls = int(pair["true_class"])
    pred_cls = int(pair["pred_class"])

    true_name = (
        class_names[true_cls] if true_cls < len(class_names) else f"Class_{true_cls}"
    )
    pred_name = (
        class_names[pred_cls] if pred_cls < len(class_names) else f"Class_{pred_cls}"
    )

    true_short = true_name.split("/")[-1] if "/" in true_name else true_name
    pred_short = pred_name.split("/")[-1] if "/" in pred_name else pred_name

    print(f"\n{'='*100}")
    print(f"Pair {pair_idx+1}: {pair['count']} misclassifications")
    print(f"  TRUE CLASS:      {true_name} (ID: {true_cls})")
    print(f"  PREDICTED AS:    {pred_name} (ID: {pred_cls})")
    print(f"{'='*100}")
    confusion_examples = []
    for idx in range(len(holdout_labels_array)):
        if (
            holdout_labels_array[idx] == true_cls
            and holdout_preds_array[idx] == pred_cls
        ):
            confusion_examples.append(final_test_indices[idx])
    train_true_cls_indices = list(label_index.get(true_cls, []))
    train_pred_cls_indices = list(label_index.get(pred_cls, []))

    n_confused = min(3, len(confusion_examples))
    n_cols = 2 + n_confused

    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 5))

    # Column 1: True class reference (from training set)
    if train_true_cls_indices:
        img = ds_train["images"][int(train_true_cls_indices[0])].numpy()
        axes[0].imshow(img)
        axes[0].set_title(
            f"TRUE CLASS (train ref)\n{true_short[:30]}",
            fontsize=11,
            fontweight="bold",
            color="green",
        )
        axes[0].set_xlabel(f"(ID: {true_cls})", fontsize=9)
        for spine in axes[0].spines.values():
            spine.set_edgecolor("green")
            spine.set_linewidth(3)
    axes[0].set_xticks([])
    axes[0].set_yticks([])

    # Column 2: Predicted class reference (from training set)
    if train_pred_cls_indices:
        img = ds_train["images"][int(train_pred_cls_indices[0])].numpy()
        axes[1].imshow(img)
        axes[1].set_title(
            f"CONFUSED WITH (train ref)\n{pred_short[:30]}",
            fontsize=11,
            fontweight="bold",
            color="red",
        )
        axes[1].set_xlabel(f"(ID: {pred_cls})", fontsize=9)
        for spine in axes[1].spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    axes[1].set_xticks([])
    axes[1].set_yticks([])

    # Columns 3+: Misclassified examples from holdout
    for i in range(n_confused):
        ax = axes[2 + i]
        if i < len(confusion_examples):
            img = ds_val["images"][int(confusion_examples[i])].numpy()
            ax.imshow(img)
            ax.set_title(
                f"MISCLASSIFIED #{i+1}\n(from holdout)", fontsize=10, color="orange"
            )
            ax.set_xlabel(f"True: {true_cls} → Pred: {pred_cls}", fontsize=8)
            for spine in ax.spines.values():
                spine.set_edgecolor("orange")
                spine.set_linewidth(2)
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

These highly confused classes generally show images of similar looking species (body size, plumage, beak shape, and colours), often with birds showing similar poses as the confused class. For example, flying birds (reference image for Foster’s Tern) can easily be confused with flying birds from a similar looking class (Common Tern).

Examining LIME explanations for the best and worst predictions can show what features help or hurt a classification:

In [ ]:
print("=" * 80)
print("LIME EXPLANATIONS: BEST vs WORST PREDICTIONS (HOLDOUT)")
print("=" * 80)
print("Using LEARNED PROJECTION model with prototype matching")

print("\nBuilding prediction confidence info from holdout evaluation...")
all_confidences = []
for i in range(len(holdout_true_labels)):
    all_confidences.append({
        'sample_idx': i,
        'ds_idx': int(final_test_indices[i]),
        'true_label': int(holdout_true_labels[i]),
        'pred_label': int(holdout_predictions[i]),
        'confidence': float(holdout_confidences[i]),
        'correct': int(holdout_true_labels[i]) == int(holdout_predictions[i])
    })
all_confidences.sort(key=lambda x: x['confidence'], reverse=True)

# Select best predictions (highest confidence, correct)
best_predictions = [c for c in all_confidences if c['correct']][:3]
for i, pred in enumerate(best_predictions):
    true_name = class_names[pred['true_label']] if pred['true_label'] < len(class_names) else f"Class_{pred['true_label']}"
    short_name = true_name.split('/')[-1] if '/' in true_name else true_name

# Select worst predictions (incorrect, or lowest confidence)
worst_incorrect = [c for c in all_confidences if not c['correct']][:3]
worst_predictions = worst_incorrect
if len(worst_predictions) < 3:
    lowest_conf_correct = [c for c in reversed(all_confidences) if c['correct']]
    worst_predictions.extend(lowest_conf_correct[:3 - len(worst_predictions)])
for i, pred in enumerate(worst_predictions):
    true_name = class_names[pred['true_label']] if pred['true_label'] < len(class_names) else f"Class_{pred['true_label']}"
    pred_name = class_names[pred['pred_label']] if pred['pred_label'] < len(class_names) else f"Class_{pred['pred_label']}"
    true_short = true_name.split('/')[-1] if '/' in true_name else true_name
    pred_short = pred_name.split('/')[-1] if '/' in pred_name else pred_name
    status = "WRONG" if not pred['correct'] else "low conf"

backbone_model = extractor.model
backbone_model.eval()
weights = models.EfficientNet_B4_Weights.IMAGENET1K_V1
preprocess = weights.transforms()
prototype_labels_sorted = sorted(prototypes.keys())
prototype_matrix_np = np.stack([prototypes[lbl] for lbl in prototype_labels_sorted])
prototype_matrix_np_norm = prototype_matrix_np / (np.linalg.norm(prototype_matrix_np, axis=1, keepdims=True) + 1e-8)
label_to_lime_idx = {lbl: i for i, lbl in enumerate(prototype_labels_sorted)}
n_classes_lime = len(prototype_labels_sorted)

print(f"\nLIME prediction function configured:")
print(f"  Prototype classes: {n_classes_lime}")
print(f"  Projection model: {model.__class__.__name__}")

# Initialize LIME explainer
explainer = lime_image.LimeImageExplainer()
print("\n" + "=" * 80)
print("LIME EXPLANATIONS: BEST PREDICTIONS")
print("=" * 80)

n_best = len(best_predictions)
if n_best > 0:
    fig, axes = plt.subplots(n_best, 4, figsize=(20, 5 * n_best))
    if n_best == 1:
        axes = axes.reshape(1, -1)

    for row, pred_info in enumerate(best_predictions):
        ds_idx = pred_info['ds_idx']
        true_label = pred_info['true_label']
        pred_label = pred_info['pred_label']
        confidence = pred_info['confidence']

        original_img = ds_val.images[ds_idx].numpy()
        img_pil = Image.fromarray(original_img)
        img_resized = np.array(img_pil.resize((224, 224)))

        try:
            explanation = explainer.explain_instance(
                img_resized,
                predict_fn_lime,
                top_labels=10,
                hide_color=0,
                num_samples=500,
                random_seed=42
            )

            species_name = class_names[true_label] if true_label < len(class_names) else f"Class_{true_label}"
            short_name = species_name.split('/')[-1] if '/' in species_name else species_name

            pred_lime_idx = label_to_lime_idx.get(pred_label, None)
            available_labels = list(explanation.local_exp.keys())

            # Column 1: Original image
            axes[row, 0].imshow(img_resized)
            axes[row, 0].set_title(f"BEST #{row+1}\n{short_name[:30]}\nConf: {confidence*100:.1f}%",
                                   fontsize=10, fontweight='bold', color='green')
            axes[row, 0].axis('off')

            # Column 2: Positive features only
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=True, num_features=5, hide_rest=False)
                axes[row, 1].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 1].set_title(f"Positive Features\n(supporting prediction)", fontsize=9)
            else:
                axes[row, 1].imshow(img_resized)
                axes[row, 1].set_title("No explanation", fontsize=9)
            axes[row, 1].axis('off')

            # Column 3: All features (positive and negative)
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=False, num_features=10, hide_rest=False)
                axes[row, 2].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 2].set_title(f"All Features\n(green=+, red=-)", fontsize=9)
            else:
                axes[row, 2].imshow(img_resized)
                axes[row, 2].set_title("No explanation", fontsize=9)
            axes[row, 2].axis('off')

            # Column 4: Highlighted regions only
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=True, num_features=5, hide_rest=True)
                axes[row, 3].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 3].set_title(f"Key Regions Only\n(rest hidden)", fontsize=9)
            else:
                axes[row, 3].imshow(img_resized)
                axes[row, 3].set_title("No explanation", fontsize=9)
            axes[row, 3].axis('off')

            print(f"[OK] Best prediction #{row+1}: {short_name} ({confidence*100:.1f}%)")

        except Exception as e:
            print(f"LIME failed for best #{row+1}: {e}")
            for col in range(4):
                axes[row, col].imshow(img_resized)
                axes[row, col].set_title("Error", fontsize=9)
                axes[row, col].axis('off')

    plt.suptitle('LIME Explanations: BEST Predictions (High Confidence, Correct)',
                 fontsize=14, fontweight='bold', y=1.02, color='green')
    plt.tight_layout()
    plt.show()

# Generate LIME explanations for worst predictions
print("\n" + "=" * 80)
print("LIME EXPLANATIONS: WORST PREDICTIONS")
print("=" * 80)

n_worst = len(worst_predictions)
if n_worst > 0:
    fig, axes = plt.subplots(n_worst, 4, figsize=(20, 5 * n_worst))
    if n_worst == 1:
        axes = axes.reshape(1, -1)

    for row, pred_info in enumerate(worst_predictions):
        ds_idx = pred_info['ds_idx']
        true_label = pred_info['true_label']
        pred_label = pred_info['pred_label']
        confidence = pred_info['confidence']
        is_correct = pred_info['correct']

        original_img = ds_val.images[ds_idx].numpy()
        img_pil = Image.fromarray(original_img)
        img_resized = np.array(img_pil.resize((224, 224)))

        try:
            explanation = explainer.explain_instance(
                img_resized,
                predict_fn_lime,
                top_labels=10,
                hide_color=0,
                num_samples=500,
                random_seed=42
            )

            true_name = class_names[true_label] if true_label < len(class_names) else f"Class_{true_label}"
            pred_name = class_names[pred_label] if pred_label < len(class_names) else f"Class_{pred_label}"
            true_short = true_name.split('/')[-1] if '/' in true_name else true_name
            pred_short = pred_name.split('/')[-1] if '/' in pred_name else pred_name

            pred_lime_idx = label_to_lime_idx.get(pred_label, None)
            available_labels = list(explanation.local_exp.keys())

            # Column 1: Original image with error info
            axes[row, 0].imshow(img_resized)
            status = "WRONG" if not is_correct else "Low Conf"
            axes[row, 0].set_title(f"WORST #{row+1} [{status}]\nTrue: {true_short[:20]}\nPred: {pred_short[:20]} ({confidence*100:.1f}%)",
                                   fontsize=9, fontweight='bold', color='red')
            axes[row, 0].axis('off')

            # Column 2: Positive features only (for predicted class)
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=True, num_features=5, hide_rest=False)
                axes[row, 1].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 1].set_title(f"Positive Features\n(supporting prediction)", fontsize=9)
            else:
                axes[row, 1].imshow(img_resized)
                axes[row, 1].set_title("No explanation", fontsize=9)
            axes[row, 1].axis('off')

            # Column 3: All features (positive and negative)
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=False, num_features=10, hide_rest=False)
                axes[row, 2].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 2].set_title(f"All Features\n(green=+, red=-)", fontsize=9)
            else:
                axes[row, 2].imshow(img_resized)
                axes[row, 2].set_title("No explanation", fontsize=9)
            axes[row, 2].axis('off')

            # Column 4: Highlighted regions only
            if pred_lime_idx is not None and pred_lime_idx in available_labels:
                temp, mask = explanation.get_image_and_mask(pred_lime_idx, positive_only=True, num_features=5, hide_rest=True)
                axes[row, 3].imshow(mark_boundaries(temp / 255.0, mask))
                axes[row, 3].set_title(f"Key Regions Only\n(rest hidden)", fontsize=9)
            else:
                axes[row, 3].imshow(img_resized)
                axes[row, 3].set_title("No explanation", fontsize=9)
            axes[row, 3].axis('off')

            print(f"[OK] Worst prediction #{row+1}: True={true_short[:20]}, Pred={pred_short[:20]} ({confidence*100:.1f}%)")

        except Exception as e:
            print(f"LIME failed for worst #{row+1}: {e}")
            for col in range(4):
                axes[row, col].imshow(img_resized)
                axes[row, col].set_title("Error", fontsize=9)
                axes[row, col].axis('off')

    plt.suptitle('LIME Explanations: WORST Predictions (Incorrect or Low Confidence)',
                 fontsize=14, fontweight='bold', y=1.02, color='red')
    plt.tight_layout()
    plt.show()

The LIME plot shows the regions that are the most important in the example’s prediction. The green regions are areas that support the class prediction, and the red regions are regions that contradict the prediction. The model's top 3 predicsions were all of the American Coot, where it's prediction was correct and confident. The model correctly identified the bird's head and parts of its body. All three sample images show the bird swimming against a uniform background. The model also uses the part of the water background to help these predictions, which indicates that these high confidence and correct predictions likely would not generalize to images of the same species bird with a different background, or of the bird flying. The model's worst predictions showed high confidence in the wrong species. Areas of the image supporting these decisions all include the whole bird, which indicates that the model is still correctly identifying the bird in the image. In each example image, the model is also using parts of the background to support its decision.

Showing final results as a json:

In [ ]:
support_set_size = sum(
    len(indices) for indices in experiment_lp.support_indices.values()
)
initial_support_size = (
    sum(len(indices) for indices in support_indices.values())
    if "support_indices" in dir()
    else support_set_size
)
pool_remaining = (
    sum(len(indices) for indices in pool_indices.values())
    if "pool_indices" in dir()
    else 0
)
n_epochs = globals().get("N_EPOCHS", 30)
fewshot_per_class = globals().get("FEWSHOT_PER_CLASS", 5)
final_summary = {
    "timestamp": datetime.now().isoformat(),
    "experiment": "Learned Projection with Pseudo-Labeling for Bird Classification",
    "config": {
        "backbone": "efficientnet_b4",
        "preprocess_mode": "bbox_crop",
        "n_support_initial": fewshot_per_class,
        "n_epochs": n_epochs,
        "embedding_dim": 1792,
        "projection_dim": 512,
        "n_classes": NUM_CLASSES,
    },
    "results": {
        "frozen_baseline": 0.5088,
        "learned_projection_holdout": float(accuracy),
    },
    "holdout_metrics": {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    },
    "data_stats": {
        "initial_support_set_size": initial_support_size,
        "final_support_set_size": support_set_size,
        "pseudo_labeled_samples": support_set_size - initial_support_size,
        "pool_remaining": pool_remaining,
        "holdout_test_samples": len(final_test_labels),
        "n_classes": len(np.unique(final_test_labels)),
    },
    "per_class_stats": {
        "mean_accuracy": float(np.mean(holdout_accuracies)),
        "median_accuracy": float(np.median(holdout_accuracies)),
        "std_accuracy": float(np.std(holdout_accuracies)),
        "classes_zero_acc": int(sum(1 for a in holdout_accuracies if a == 0)),
        "classes_perfect_acc": int(sum(1 for a in holdout_accuracies if a == 1.0)),
        "classes_above_70": int(classes_above_70),
    },
    "total_improvement": float(accuracy - 0.5088),
}

results_path = DATA_ROOT / "learned_projection_holdout_results.json"
with open(results_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print(f"Results saved to {results_path}")
print(f"\n{'='*70}")
print("FINAL SUMMARY")
print(f"{'='*70}")
print(json.dumps(final_summary, indent=2))

# Conclusion

In conclusion, this notebook illustrates a few shot learning approach for the NABirds dataset, which contains 555 North American bird species. To begin, a maximum of 5 samples per class were used to build a prototype based classifier comparing embeddings from EfficientNet-B4, ResNet-50, and VIT-B-16. Overall, the EfficientNet model using bbox processing with a learned projection head (PrototypicalNetwork with 512 learned embeddings) obtained the highest baseline accuracy (approximately 60%). By pseudo labeling samples with high predicted confidence, and carefully examining some sample example images against a reference image, the training data was able to grow from close to 3000 samples to almost 13000 samples. By periodically retraining the model and using the model to evaluate new samples, accuracy was able to increase to almost 72%, which is slightly above the target for this task. Precision was 70%,  recall was 69%, and F1 score was 68%. The model frequently confused pairs of the different sub-species (such as the Red-shafted and Yellow-shafted Northern Flicker), though most classes had accuracy over 70%.


Future work:
- Enabling hierarchical classification to reduce subspecies or similar species confusion could help to improve model accuracy. For example, classifying a general species, then a genus could help to reduce inter-class confusion.
- Adding attention layers to help the model focus on regions of the birds could increase accuracy by reducing background focus.
- To help model predictions, a more robust pseudo labeling approach could be used, combining confidence and distance between prototypes, or by limiting the number of samples added for each pseudo labeling iteration.
- Using data augmentation could help to reduce confusion between species that look similar, or that have similar backgrounds.

In the next notebook, the model will be tuned on the entire labeled dataset.